In [1]:
import faulthandler
crash_log = open("crash.log", "w")          # keep this reference alive
faulthandler.enable(file=crash_log, all_threads=True)

In [2]:
%matplotlib widget
import torch

from thesis_code.plotting import plot_performance, plot_gradients

In [3]:
# flushing = torch.set_flush_denormal(True)
# print(f"Floating point flushing is set to {flushing}")

torch.manual_seed(42)
dtype = torch.float
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


In [4]:
dt = 0.001

In [5]:
import tonic
from tonic import DiskCachedDataset
import tonic.transforms as transforms
from torch.utils.data import DataLoader
import shutil
from pathlib import Path

PROJECT_ROOT = Path("C:\\Users\\emiel\\Documents\\Study\\Thesis\\Thesis-Code")
DATASETS_DIR = PROJECT_ROOT / "datasets"

def get_SHD_dataloader(batch_size: int, time_window: float = 1000, train: bool = True, shuffle: bool = True, drop_last: bool = False, re_download: bool = False) -> tuple[DataLoader, int, int]:
    train_extension = "train" if train else "test"
    cache_path = DATASETS_DIR / "cache" / "SHD" / train_extension
    data_path = DATASETS_DIR / "data"

    if re_download:
        shutil.rmtree(cache_path, ignore_errors=True)
        shutil.rmtree(DATASETS_DIR / "data" / "SHD", ignore_errors=True)

    sensor_size = tonic.datasets.SHD.sensor_size
    frame_transform=transforms.ToFrame(
        sensor_size=sensor_size,
        time_window=time_window,
        start_time=0,
    )
    dataset = tonic.datasets.SHD(
        save_to=str(data_path), 
        train=train,
        transform=frame_transform,
    )
    cached_dataset = DiskCachedDataset(
        dataset,
        cache_path=str(cache_path)
    )
    dataloader = DataLoader(
        cached_dataset,
        batch_size=batch_size,
        collate_fn=tonic.collation.PadTensors(batch_first=False),
        shuffle=shuffle,
        drop_last=drop_last,
    )
    return dataloader, sensor_size[0], dataset.classes


SHD_trainloader, input_dim, classes = get_SHD_dataloader(
    batch_size=128,
    time_window=1000,
    train=True,
    shuffle=True,
    drop_last=True,
    re_download=False,
)

SHD_testloader, _, _ = get_SHD_dataloader(
    batch_size=128,
    time_window=1000,
    train=False,
    shuffle=True,
    drop_last=True,
    re_download=False,
)

In [5]:
import torch
import torch.nn as nn
from snntorch.surrogate import atan
from typing import Any

import snntorch as snn

class OutputLayer(nn.Module):
    def __init__(self, n_in: int, n_out: int, beta: float) -> None:
        super().__init__()
        self.weights = nn.Linear(n_in, n_out, bias=False)
        self.neurons = snn.Leaky(beta=beta, init_hidden=False, reset_mechanism='none')
        self.mem = self.neurons.init_leaky()

    def reset(self) -> None:
        self.mem = self.neurons.init_leaky()

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        x = self.weights(x)
        spk, self.mem = self.neurons(x, self.mem)
        return spk, self.mem

class KaimingLinear(nn.Linear):
    def __init__(self, n_in: int, n_out: int) -> None:
        super().__init__(n_in, n_out, bias=False)
        nn.init.kaiming_uniform_(self.weight, mode='fan_in', nonlinearity='relu')

class MaskedLinear(nn.Module):
    def __init__(self, con_matrix: torch.Tensor) -> None:
        super().__init__()
        self.linear = nn.Linear(*con_matrix.shape, bias=False)      # clamp to higher than 0 + small value, cannot become negative
        self.register_buffer('con_matrix', con_matrix)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return nn.functional.linear(x, self.linear.weight * self.con_matrix.T, self.linear.bias)

class MaskedKaimingLinear(nn.Module):
    def __init__(self, con_matrix: torch.Tensor) -> None:
        super().__init__()
        self.linear = KaimingLinear(*con_matrix.shape)
        self.register_buffer('con_matrix', con_matrix)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return nn.functional.linear(x, self.linear.weight * self.con_matrix.T, self.linear.bias)

class MaskedRLeaky(snn.RLeaky):
    def __init__(self, *args, rec_matrix: torch.Tensor, **kwargs) -> None:
        assert rec_matrix.shape[0] == rec_matrix.shape[1], \
            f"expected a square matrix, got shape {tuple(rec_matrix.shape)}"

        super().__init__(*args, **kwargs)
        self.recurrent = MaskedKaimingLinear(rec_matrix)
        if not self.learn_recurrent:
            for param in self.recurrent.parameters():
                param.requires_grad = False

class StandardRecurrentLayer(nn.Module):
    def __init__(self,
        forward_matrix: torch.Tensor,
        recurrent_matrix: torch.Tensor,
        beta: torch.Tensor | float,
        spike_grad: Any,
        fully_learnable: bool = True,
        record: bool = True,
    ) -> None:
        n_neurons = recurrent_matrix.shape[0]

        assert forward_matrix.shape[1] == n_neurons, \
            f"forward_matrix outputs {forward_matrix.shape[1]} neurons, recurrent_matrix has {n_neurons}"

        super().__init__()
        self.weights = MaskedKaimingLinear(forward_matrix)
        self.neurons = MaskedRLeaky(
            beta=beta,
            linear_features=n_neurons,
            init_hidden=False,
            spike_grad=spike_grad,
            learn_beta=fully_learnable,
            rec_matrix=recurrent_matrix,
        )
        self.spk, self.mem = self.neurons.init_rleaky()
        self.spk_rec, self.mem_rec = [], []
        self.record = record

    def reset(self) -> None:
        self.spk, self.mem = self.neurons.init_rleaky()
        self.spk_rec, self.mem_rec = [], []

    def get_recordings(self) -> tuple[torch.Tensor, torch.Tensor]:
        assert self.record, "recording is disabled (record=False)"
        assert self.spk_rec, "no recordings yet; run at least one forward step"
        return torch.stack(self.spk_rec), torch.stack(self.mem_rec)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.weights(x)
        self.spk, self.mem = self.neurons(x, self.spk, self.mem)
        if self.record:
            self.spk_rec.append(self.spk)
            self.mem_rec.append(self.mem)
        return self.spk

class Recorder:
    def __init__(self, num_layers: int) -> None:
        self.num_layers = num_layers
        self.num_iterations = 0
        self.clear()

    def clear(self) -> None:
        self.recordings = {i: {"spk_recs": [], "mem_recs": []} for i in range(self.num_layers)}

    def increment_iterations(self) -> None:
        self.num_iterations += 1

    def add_layer_recordings(self, layer_i: int, spk_rec: torch.Tensor, mem_rec: torch.Tensor) -> None:
        self.recordings[layer_i]["spk_recs"].append(spk_rec.detach().cpu())
        self.recordings[layer_i]["mem_recs"].append(mem_rec.detach().cpu())

    def get_recordings(self, layer_i: int, iteration_i: int) -> tuple[torch.Tensor, torch.Tensor]:
        spk_rec = self.recordings[layer_i]["spk_recs"][iteration_i]
        mem_rec = self.recordings[layer_i]["mem_recs"][iteration_i]
        return spk_rec, mem_rec

class StandardSRNN(nn.Module):
    def __init__(self, 
            forward_matrices: list[torch.Tensor],
            recurrent_matrices: list[torch.Tensor],
            betas: list[torch.Tensor | float],
            beta_out: float,
            n_classes: int,
            spike_grad: Any = atan(),
            fully_learnable: bool = True,
            record: bool = True,
    ) -> None:
        super().__init__()

        self.rec_layers = nn.ModuleList()

        assert len(forward_matrices) == len(recurrent_matrices) == len(betas), \
            "unequal number forward- and recurrent matrices and betas"

        for fm_prev, fm_next in zip(forward_matrices, forward_matrices[1:]):
            assert fm_prev.shape[1] == fm_next.shape[0], \
                f"layer output {fm_prev.shape[1]} doesn't match next layer input {fm_next.shape[0]}"
        
        for fm, rm, beta in zip(forward_matrices, recurrent_matrices, betas):
            assert fm.shape[-1] == rm.shape[0], \
                f"forward matrix outputs {fm.shape[-1]} neurons, recurrent matrix has {rm.shape[0]}"

            self.rec_layers.append(StandardRecurrentLayer(
                forward_matrix=fm,
                recurrent_matrix=rm,
                beta=beta,
                spike_grad=spike_grad,
                fully_learnable=fully_learnable,
                record=record,
            ))

        self.out_layer = OutputLayer(forward_matrices[-1].shape[-1], n_classes, beta_out)
        self.recorder = Recorder(num_layers=len(recurrent_matrices))
        self.record = record

    def reset(self) -> None:
        for layer in self.rec_layers:
            layer.reset()
        self.out_layer.reset()

    def record_layers(self) -> None:
        for i, layer in enumerate(self.rec_layers):
            self.recorder.add_layer_recordings(i, *layer.get_recordings())
        self.recorder.increment_iterations()

    def get_recorder(self) -> Recorder:
        assert self.record, "recording is disabled (record=False)"
        return self.recorder

    def forward(self, data: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, list[torch.Tensor]]:
        spk_outs, mem_outs = [], []
        hidden_spks = [[] for _ in range(len(self.rec_layers))]
        self.reset()

        for step in range(data.size(0)):
            x = data[step]
            for i, layer in enumerate(self.rec_layers):
                x = layer(x)
                hidden_spks[i].append(x)

            spk_out, mem_out = self.out_layer(x)
            spk_outs.append(spk_out)
            mem_outs.append(mem_out)

        if self.record:
            self.record_layers()

        return torch.stack(spk_outs, dim=0), torch.stack(mem_outs, dim=0), [torch.stack(spks_h, dim=0) for spks_h in hidden_spks]

In [6]:
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import numpy as np
import torch.nn.functional as F
from snntorch import surrogate
from typing import Callable

def get_compute_loss_reg_fn(lam_lower: float, v_lower: float, lam_uppers: list[float], v_uppers: list[float], L: int = 2) -> callable:
    """
    Based on:
    Friedemann Zenke, Tim P. Vogels; The Remarkable Robustness of Surrogate Gradient Learning for Instilling Complex Function in Spiking Neural Networks. Neural Comput 2021; 33 (4): 899-925. doi: https://doi.org/10.1162/neco_a_01367
    """
    assert len(lam_uppers) == len(v_uppers), \
        f"Mismatching parameter counts: lam_uppers={len(lam_uppers)}, v_uppers={len(v_uppers)}"
    
    def compute_loss_reg(hidden_spks: list[torch.Tensor]) -> torch.Tensor:
        loss_regs = []
        for spks_h, lam_upper, v_upper in zip(hidden_spks, lam_uppers, v_uppers):
            # spks_h has shape (n_steps, batch_size, n_neurons), for example: (116, 128, 256)
            spk_count = spks_h.sum(dim=0)  # has shape (batch_size, n_neurons)
            g_lower = lam_lower * torch.relu(v_lower - spk_count).square().mean(dim=-1)
            g_upper = lam_upper * torch.relu(spk_count.mean(dim=-1) - v_upper).pow(L)
            loss_regs.append((g_lower + g_upper).mean(dim=0))      # output shape should be scalar ()
        return torch.stack(loss_regs).sum()
    return compute_loss_reg

def compute_loss(mem_rec: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    probs = F.softmax(mem_rec, dim=-1)
    log_probs = torch.log(probs.mean(dim=0) + 1e-8) # to avoid log(0)
    return F.nll_loss(log_probs, targets)

def measure_accuracy(mem_rec, targets):
    _, idx = mem_rec.sum(dim=0).max(1)
    return np.mean((targets == idx).detach().cpu().numpy())

def test(net: nn.Module, device: torch.device, testloader: DataLoader, max_iters: int = None) -> float:
    with torch.no_grad():
        acc = 0
        net.eval()

        for i, (data, targets) in enumerate(testloader):
            data, targets = data.to(device), targets.to(device).long()
            data = data.squeeze(2)
            
            spk_rec, mem_rec, hidden_spks = net(data)
            acc += measure_accuracy(mem_rec, targets)
            if max_iters is not None and i == max_iters:
                break

    return acc / len(testloader)

def record_grads(net: nn.Module) -> tuple[list, list]:
    grads = {name: p.grad.detach().clone()
        for name, p in net.named_parameters() if p.grad is not None}
    if len(grads) > 0:
        flat_grads = torch.cat([t.flatten() for t in grads.values()])
        return flat_grads.max(dim=0).values.cpu().numpy(), flat_grads.mean(dim=0).cpu().numpy()
    return 0, 0

def train(net: nn.Module, device: torch.device, trainloader: DataLoader, loss_fn: Callable, lr: float = 1e-3, n_epochs: int = 1, max_iters: int = None) -> tuple[nn.Module, list, list]:
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, eps=1e-6, betas=(0.9, 0.99))

    loss_hist, acc_hist, spike_hist = [], [], []
    net.train()
    max_grad_rec = []
    avg_grad_rec = []

    for epoch in range(n_epochs):
        progress_bar = tqdm(enumerate(trainloader), total=len(trainloader), desc=f"Epoch {epoch}")
        for i, (data, targets) in progress_bar:
            data, targets = data.to(device), targets.to(device).long()
            data = data.squeeze(2)

            spk_outs, mem_outs, hidden_spks = net(data)
            loss_val = loss_fn(mem_outs, targets, hidden_spks)

            optimizer.zero_grad()
            loss_val.backward()

            # torch.nn.utils.clip_grad_value_(net.parameters(), clip_value=1e6)

            max_grad, avg_grad = record_grads(net)
            max_grad_rec.append(max_grad)
            avg_grad_rec.append(avg_grad)

            optimizer.step()

            loss_hist.append(loss_val.item())

            acc = measure_accuracy(mem_outs, targets)
            acc_hist.append(acc)

            progress_bar.set_postfix(loss=f"{loss_val.item():.2f}", acc=f"{acc * 100:.2f}%")

            if i % 25 == 0:
                elapsed = progress_bar.format_dict["elapsed"]
                tqdm.write(
                    f"Epoch {epoch}, Iteration {i} — Loss: {loss_val.item():.2f}, "
                    f"Acc: {acc*100:.2f}%, Time: {elapsed:.1f}s"
                )

            if max_iters is not None and i == max_iters:
                break

    return net, loss_hist, acc_hist, spike_hist, max_grad_rec, avg_grad_rec

def tau_to_beta(dt: float, tau: float | torch.Tensor) -> torch.Tensor:
    """
    Computes 'beta = exp(-dt / tau)', implemented from:
    Friedemann Zenke, Tim P. Vogels; The Remarkable Robustness of Surrogate Gradient Learning for Instilling Complex Function in Spiking Neural Networks. Neural Comput 2021; 33 (4): 899-925. doi: https://doi.org/10.1162/neco_a_01367
    """
    return torch.exp(-dt / torch.as_tensor(tau))

def sample_heterogeneous_decays_uniform(dt: float, n_neurons: int, tau_upper: float, tau_lower: float, device=None) -> torch.Tensor:
    taus = torch.rand(n_neurons, device=device) * (tau_upper - tau_lower) + tau_lower
    return tau_to_beta(dt, taus)

def sample_heterogeneous_decays_normal(dt: float, n_neurons: int, tau_mean: float, tau_std: float, device=None) -> torch.Tensor:
    taus = (torch.randn(n_neurons, device=device) * tau_std + tau_mean).clamp(min=dt)        # ensure no negative time constants get sampled
    return tau_to_beta(dt, taus)

ns_hidden=[256, 256]
layers = [input_dim] + ns_hidden

net = StandardSRNN(
    forward_matrices=[torch.ones(n_in, n_out) for n_in, n_out in zip(layers[:-1], layers[1:])],
    recurrent_matrices=[torch.ones(n, n) for n in ns_hidden],
    betas=[sample_heterogeneous_decays_normal(dt, n, tau_mean=10 * dt, tau_std=1 * dt, device=device) for n in ns_hidden],
    beta_out=tau_to_beta(dt, 20 * dt),
    n_classes=len(classes),
    spike_grad=surrogate.fast_sigmoid(slope=25),
).to(device)

# net = models.SynapticSRNN(
#     forward_matrices=[torch.ones(n_in, n_out) for n_in, n_out in zip(layers[:-1], layers[1:])],
#     recurrent_matrices=[torch.ones(n, n) for n in ns_hidden],
#     alphas=[sample_heterogeneous_decays_normal(dt, n, tau_mean=5 * dt, tau_std=1 * dt, device=device) for n in ns_hidden],
#     betas=[sample_heterogeneous_decays_normal(dt, n, tau_mean=10 * dt, tau_std=2 * dt, device=device) for n in ns_hidden],
#     beta_out=tau_to_beta(dt, 20 * dt),
#     n_classes=len(classes),
#     spike_grad=surrogate.fast_sigmoid(slope=25),
# ).to(device)

compute_loss_reg = get_compute_loss_reg_fn(
    lam_lower=100.0,
    v_lower=1e-2,               # min n spikes
    lam_uppers=[0.5, 0.5],
    v_uppers=[50, 50],          # max n spikes per layer
    L=2,
)

accuracy = test(
    net=net,
    device=device,
    testloader=SHD_testloader,
    max_iters=None,
)
print(f"Test Accuracy before training is: {accuracy}")

net, loss_hist, acc_hist, spike_hist, max_grad_rec, avg_grad_rec = train(
    net=net,
    device=device,
    trainloader=SHD_trainloader,
    loss_fn=lambda mem_outs, targets, hidden_spks: compute_loss(mem_outs, targets) + compute_loss_reg(hidden_spks),
    lr=1e-3,
    n_epochs=50,
    max_iters=None,
)
plot_performance(loss_hist, acc_hist)
plot_gradients(max_grad_rec, avg_grad_rec)

accuracy = test(
    net=net,
    device=device,
    testloader=SHD_testloader,
    max_iters=None,
)
print(f"Test Accuracy after training is: {accuracy}")

Test Accuracy before training is: 0.0390625


Epoch 0:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 0, Iteration 0 — Loss: 2.98, Acc: 5.47%, Time: 6.8s
Epoch 0, Iteration 25 — Loss: 2.92, Acc: 11.72%, Time: 218.8s
Epoch 0, Iteration 50 — Loss: 2.86, Acc: 17.19%, Time: 418.0s


Epoch 1:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 1, Iteration 0 — Loss: 2.88, Acc: 14.06%, Time: 7.3s
Epoch 1, Iteration 25 — Loss: 2.77, Acc: 25.00%, Time: 203.9s
Epoch 1, Iteration 50 — Loss: 2.74, Acc: 25.00%, Time: 416.6s


Epoch 2:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 2, Iteration 0 — Loss: 2.70, Acc: 30.47%, Time: 9.5s


: 

In [9]:
import torch
import torch.nn as nn
from snntorch.surrogate import atan
from typing import Any

import snntorch as snn

class OutputLayer(nn.Module):
    def __init__(self, n_in: int, n_out: int, beta: float) -> None:
        super().__init__()
        self.weights = nn.Linear(n_in, n_out, bias=False)
        self.neurons = snn.Leaky(beta=beta, init_hidden=False, reset_mechanism='none')
        self.mem = self.neurons.init_leaky()

    def reset(self) -> None:
        self.mem = self.neurons.init_leaky()

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        x = self.weights(x)
        spk, self.mem = self.neurons(x, self.mem)
        return spk, self.mem

class Mask(nn.Module):
    def __init__(self, mask):
        super().__init__()
        self.register_buffer("mask", mask.to(torch.float32))

    def forward(self, w):
        return w * self.mask

class KaimingLinear(nn.Linear):
    def __init__(self, n_in: int, n_out: int) -> None:
        super().__init__(n_in, n_out, bias=False)
        nn.init.kaiming_uniform_(self.weight, mode='fan_in', nonlinearity='relu')

class MaskedLinear(nn.Module):
    def __init__(self, con_matrix: torch.Tensor) -> None:
        super().__init__()
        self.linear = nn.Linear(*con_matrix.shape, bias=False)      # clamp to higher than 0 + small value, cannot become negative
        nn.utils.parametrize.register_parametrization(self.linear, "weight", Mask(con_matrix.T.contiguous()))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x)

class MaskedKaimingLinear(nn.Module):
    def __init__(self, con_matrix: torch.Tensor) -> None:
        super().__init__()
        self.linear = KaimingLinear(*con_matrix.shape)
        nn.utils.parametrize.register_parametrization(self.linear, "weight", Mask(con_matrix.T.contiguous()))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x)

class MaskedRLeaky(snn.RLeaky):
    def __init__(self, *args, rec_matrix: torch.Tensor, **kwargs) -> None:
        assert rec_matrix.shape[0] == rec_matrix.shape[1], \
            f"expected a square matrix, got shape {tuple(rec_matrix.shape)}"

        super().__init__(*args, **kwargs)
        self.recurrent = MaskedKaimingLinear(rec_matrix)
        if not self.learn_recurrent:
            for param in self.recurrent.parameters():
                param.requires_grad = False

class StandardRecurrentLayer(nn.Module):
    def __init__(self,
        forward_matrix: torch.Tensor,
        recurrent_matrix: torch.Tensor,
        beta: torch.Tensor | float,
        spike_grad: Any,
        fully_learnable: bool = True,
        record: bool = True,
    ) -> None:
        n_neurons = recurrent_matrix.shape[0]

        assert forward_matrix.shape[1] == n_neurons, \
            f"forward_matrix outputs {forward_matrix.shape[1]} neurons, recurrent_matrix has {n_neurons}"

        super().__init__()
        self.weights = MaskedKaimingLinear(forward_matrix)
        self.neurons = MaskedRLeaky(
            beta=beta,
            linear_features=n_neurons,
            init_hidden=False,
            spike_grad=spike_grad,
            learn_beta=fully_learnable,
            rec_matrix=recurrent_matrix,
        )
        self.spk, self.mem = self.neurons.init_rleaky()
        self.spk_rec, self.mem_rec = [], []
        self.record = record

    def reset(self) -> None:
        self.spk, self.mem = self.neurons.init_rleaky()
        self.spk_rec, self.mem_rec = [], []

    def get_recordings(self) -> tuple[torch.Tensor, torch.Tensor]:
        assert self.record, "recording is disabled (record=False)"
        assert self.spk_rec, "no recordings yet; run at least one forward step"
        return torch.stack(self.spk_rec), torch.stack(self.mem_rec)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.weights(x)
        self.spk, self.mem = self.neurons(x, self.spk, self.mem)
        if self.record:
            self.spk_rec.append(self.spk)
            self.mem_rec.append(self.mem)
        return self.spk

class Recorder:
    def __init__(self, num_layers: int) -> None:
        self.num_layers = num_layers
        self.num_iterations = 0
        self.clear()

    def clear(self) -> None:
        self.recordings = {i: {"spk_recs": [], "mem_recs": []} for i in range(self.num_layers)}

    def increment_iterations(self) -> None:
        self.num_iterations += 1

    def add_layer_recordings(self, layer_i: int, spk_rec: torch.Tensor, mem_rec: torch.Tensor) -> None:
        self.recordings[layer_i]["spk_recs"].append(spk_rec.detach().cpu())
        self.recordings[layer_i]["mem_recs"].append(mem_rec.detach().cpu())

    def get_recordings(self, layer_i: int, iteration_i: int) -> tuple[torch.Tensor, torch.Tensor]:
        spk_rec = self.recordings[layer_i]["spk_recs"][iteration_i]
        mem_rec = self.recordings[layer_i]["mem_recs"][iteration_i]
        return spk_rec, mem_rec

class StandardSRNNSpedUp(nn.Module):
    def __init__(self, 
            forward_matrices: list[torch.Tensor],
            recurrent_matrices: list[torch.Tensor],
            betas: list[torch.Tensor | float],
            beta_out: float,
            n_classes: int,
            spike_grad: Any = atan(),
            fully_learnable: bool = True,
            record: bool = True,
    ) -> None:
        super().__init__()

        self.rec_layers = nn.ModuleList()

        assert len(forward_matrices) == len(recurrent_matrices) == len(betas), \
            "unequal number forward- and recurrent matrices and betas"

        for fm_prev, fm_next in zip(forward_matrices, forward_matrices[1:]):
            assert fm_prev.shape[1] == fm_next.shape[0], \
                f"layer output {fm_prev.shape[1]} doesn't match next layer input {fm_next.shape[0]}"
        
        for fm, rm, beta in zip(forward_matrices, recurrent_matrices, betas):
            assert fm.shape[-1] == rm.shape[0], \
                f"forward matrix outputs {fm.shape[-1]} neurons, recurrent matrix has {rm.shape[0]}"

            self.rec_layers.append(StandardRecurrentLayer(
                forward_matrix=fm,
                recurrent_matrix=rm,
                beta=beta,
                spike_grad=spike_grad,
                fully_learnable=fully_learnable,
                record=record,
            ))

        self.out_layer = OutputLayer(forward_matrices[-1].shape[-1], n_classes, beta_out)
        self.recorder = Recorder(num_layers=len(recurrent_matrices))
        self.record = record

    def reset(self) -> None:
        for layer in self.rec_layers:
            layer.reset()
        self.out_layer.reset()

    def record_layers(self) -> None:
        for i, layer in enumerate(self.rec_layers):
            self.recorder.add_layer_recordings(i, *layer.get_recordings())
        self.recorder.increment_iterations()

    def get_recorder(self) -> Recorder:
        assert self.record, "recording is disabled (record=False)"
        return self.recorder

    def forward(self, data: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, list[torch.Tensor]]:
        spk_outs, mem_outs = [], []
        hidden_spks = [[] for _ in range(len(self.rec_layers))]
        self.reset()

        with nn.utils.parametrize.cached():
            for step in range(data.size(0)):
                x = data[step]
                for i, layer in enumerate(self.rec_layers):
                    x = layer(x)
                    hidden_spks[i].append(x)

                spk_out, mem_out = self.out_layer(x)
                spk_outs.append(spk_out)
                mem_outs.append(mem_out)

        if self.record:
            self.record_layers()

        return torch.stack(spk_outs, dim=0), torch.stack(mem_outs, dim=0), [torch.stack(spks_h, dim=0) for spks_h in hidden_spks]


import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import numpy as np
import torch.nn.functional as F
from snntorch import surrogate
from typing import Callable

def get_compute_loss_reg_fn(lam_lower: float, v_lower: float, lam_uppers: list[float], v_uppers: list[float], L: int = 2) -> callable:
    """
    Based on:
    Friedemann Zenke, Tim P. Vogels; The Remarkable Robustness of Surrogate Gradient Learning for Instilling Complex Function in Spiking Neural Networks. Neural Comput 2021; 33 (4): 899-925. doi: https://doi.org/10.1162/neco_a_01367
    """
    assert len(lam_uppers) == len(v_uppers), \
        f"Mismatching parameter counts: lam_uppers={len(lam_uppers)}, v_uppers={len(v_uppers)}"
    
    def compute_loss_reg(hidden_spks: list[torch.Tensor]) -> torch.Tensor:
        loss_regs = []
        for spks_h, lam_upper, v_upper in zip(hidden_spks, lam_uppers, v_uppers):
            # spks_h has shape (n_steps, batch_size, n_neurons), for example: (116, 128, 256)
            spk_count = spks_h.sum(dim=0)  # has shape (batch_size, n_neurons)
            g_lower = lam_lower * torch.relu(v_lower - spk_count).square().mean(dim=-1)
            g_upper = lam_upper * torch.relu(spk_count.mean(dim=-1) - v_upper).pow(L)
            loss_regs.append((g_lower + g_upper).mean(dim=0))      # output shape should be scalar ()
        return torch.stack(loss_regs).sum()
    return compute_loss_reg

def compute_loss(mem_rec: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    probs = F.softmax(mem_rec, dim=-1)
    log_probs = torch.log(probs.mean(dim=0) + 1e-8) # to avoid log(0)
    return F.nll_loss(log_probs, targets)

def measure_accuracy(mem_rec: torch.Tensor, targets: torch.Tensor) -> np.ndarray:
    _, idx = mem_rec.sum(dim=0).max(1)
    return np.mean((targets == idx).detach().cpu().numpy())

def test(net: nn.Module, device: torch.device, testloader: DataLoader, max_iters: int = None) -> float:
    with torch.no_grad():
        acc = 0
        net.eval()

        for i, (data, targets) in enumerate(testloader):
            data, targets = data.to(device), targets.to(device).long()
            data = data.squeeze(2)
            
            spk_rec, mem_rec, hidden_spks = net(data)
            acc += measure_accuracy(mem_rec, targets)
            if max_iters is not None and i == max_iters:
                break

    return acc / len(testloader)

def record_grads(net: nn.Module) -> tuple[list, list]:
    grads = {name: p.grad.detach().clone()
        for name, p in net.named_parameters() if p.grad is not None}
    if len(grads) > 0:
        flat_grads = torch.cat([t.flatten() for t in grads.values()])
        return flat_grads.max(dim=0).values.cpu().numpy(), flat_grads.mean(dim=0).cpu().numpy()
    return 0, 0

import psutil, os

class PerformanceRecorder:
    def __init__(self) -> None:
        self.loss_rec = []
        self.acc_rec = []

    def record(self, loss_val: float, acc: float) -> None:
        self.loss_rec.append(loss_val)
        self.acc_rec.append(acc)

    def get_performance(self) -> tuple[list, list]:
        return self.loss_rec, self.acc_rec

class GradientRecorder:
    def __init__(self) -> None:
        self.max_grad_rec = []
        self.avg_grad_rec = []

    def record(self, net: nn.Module) -> None:
        grads = {name: p.grad.detach().clone()
            for name, p in net.named_parameters() if p.grad is not None}
        if len(grads) > 0:
            flat_grads = torch.cat([t.flatten() for t in grads.values()])
            self.max_grad_rec.append(flat_grads.max(dim=0).values.cpu().numpy())
            self.avg_grad_rec.append(flat_grads.mean(dim=0).cpu().numpy())
        else:
            self.max_grad_rec.append(0)
            self.avg_grad_rec.append(0)

    def get_grads(self) -> tuple[list, list]:
        return self.max_grad_rec, self.avg_grad_rec

def train(net: nn.Module, device: torch.device, trainloader: DataLoader, loss_fn: Callable, lr: float = 1e-3, n_epochs: int = 1, max_iters: int = None) -> tuple[nn.Module, PerformanceRecorder, GradientRecorder]:
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    process = psutil.Process(os.getpid())
    perf_rec = PerformanceRecorder()
    grad_rec = GradientRecorder()

    net.train()

    for epoch in range(n_epochs):
        progress_bar = tqdm(enumerate(trainloader), total=len(trainloader), desc=f"Epoch {epoch}")
        losses, accs = [], []
        for i, (data, targets) in progress_bar:
            data, targets = data.to(device), targets.to(device).long()
            data = data.squeeze(2)

            spk_outs, mem_outs, hidden_spks = net(data)
            loss_val = loss_fn(mem_outs, targets, hidden_spks)

            optimizer.zero_grad()
            loss_val.backward()
            grad_rec.record(net)
            optimizer.step()

            losses.append(loss_val.item())
            acc = measure_accuracy(mem_outs.detach(), targets)
            accs.append(acc)

            progress_bar.set_postfix(
                loss=f"{loss_val.item():.2f}",
                acc=f"{acc * 100:.2f}%",
                iteration=f"\nIteration {i}",
                time_elapsed=f"Time elapsed: {progress_bar.format_dict['elapsed']:.1f}s",
                used_memory=f"{process.memory_info().rss / 1e9:.2f} GB",
            )

            if max_iters is not None and i == max_iters:
                break

        perf_rec.record(float(np.mean(losses)), float(np.mean(accs)))
    return net, perf_rec, grad_rec

def tau_to_beta(dt: float, tau: float | torch.Tensor) -> torch.Tensor:
    """
    Computes 'beta = exp(-dt / tau)', implemented from:
    Friedemann Zenke, Tim P. Vogels; The Remarkable Robustness of Surrogate Gradient Learning for Instilling Complex Function in Spiking Neural Networks. Neural Comput 2021; 33 (4): 899-925. doi: https://doi.org/10.1162/neco_a_01367
    """
    return torch.exp(-dt / torch.as_tensor(tau))

def sample_heterogeneous_decays_uniform(dt: float, n_neurons: int, tau_upper: float, tau_lower: float, device=None) -> torch.Tensor:
    taus = torch.rand(n_neurons, device=device) * (tau_upper - tau_lower) + tau_lower
    return tau_to_beta(dt, taus)

def sample_heterogeneous_decays_normal(dt: float, n_neurons: int, tau_mean: float, tau_std: float, device=None) -> torch.Tensor:
    taus = (torch.randn(n_neurons, device=device) * tau_std + tau_mean).clamp(min=dt)        # ensure no negative time constants get sampled
    return tau_to_beta(dt, taus)

ns_hidden=[256, 256]
layers = [input_dim] + ns_hidden

net = StandardSRNNSpedUp(
    forward_matrices=[torch.ones(n_in, n_out) for n_in, n_out in zip(layers[:-1], layers[1:])],
    recurrent_matrices=[torch.ones(n, n) for n in ns_hidden],
    betas=[sample_heterogeneous_decays_normal(dt, n, tau_mean=10 * dt, tau_std=1 * dt, device=device) for n in ns_hidden],
    beta_out=tau_to_beta(dt, 20 * dt),
    n_classes=len(classes),
    spike_grad=surrogate.fast_sigmoid(slope=25),
    record=False,
).to(device)

# net = models.SynapticSRNN(
#     forward_matrices=[torch.ones(n_in, n_out) for n_in, n_out in zip(layers[:-1], layers[1:])],
#     recurrent_matrices=[torch.ones(n, n) for n in ns_hidden],
#     alphas=[sample_heterogeneous_decays_normal(dt, n, tau_mean=5 * dt, tau_std=1 * dt, device=device) for n in ns_hidden],
#     betas=[sample_heterogeneous_decays_normal(dt, n, tau_mean=10 * dt, tau_std=2 * dt, device=device) for n in ns_hidden],
#     beta_out=tau_to_beta(dt, 20 * dt),
#     n_classes=len(classes),
#     spike_grad=surrogate.fast_sigmoid(slope=25),
# ).to(device)

compute_loss_reg = get_compute_loss_reg_fn(
    lam_lower=100.0,
    v_lower=1e-2,               # min n spikes
    lam_uppers=[0.5, 0.5],
    v_uppers=[50, 50],          # max n spikes per layer
    L=2,
)

accuracy = test(
    net=net,
    device=device,
    testloader=SHD_testloader,
    max_iters=None,
)
print(f"Test Accuracy before training is: {accuracy}")

net, loss_hist, acc_hist, spike_hist, max_grad_rec, avg_grad_rec = train(
    net=net,
    device=device,
    trainloader=SHD_trainloader,
    loss_fn=lambda mem_outs, targets, hidden_spks: compute_loss(mem_outs, targets) + compute_loss_reg(hidden_spks),
    lr=1e-3,
    n_epochs=50,
    max_iters=None,
)
plot_performance(loss_hist, acc_hist)
plot_gradients(max_grad_rec, avg_grad_rec)

accuracy = test(
    net=net,
    device=device,
    testloader=SHD_testloader,
    max_iters=None,
)
print(f"Test Accuracy after training is: {accuracy}")

Test Accuracy before training is: 0.03722426470588235


Epoch 0:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 1:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/63 [00:00<?, ?it/s]

KeyboardInterrupt: 